In [3]:
import pandas as pd

DATA_DIR = "data/processed"
df = pd.read_parquet(f"{DATA_DIR}/aqi_features_v1.parquet")

# Load feature columns
FEATURE_COLS = pd.read_json(f"{DATA_DIR}/feature_columns_v1.json", typ='series').tolist()
TARGET_COLS = ["pm25_t_plus_24h", "pm25_t_plus_48h", "pm25_t_plus_72h"]

X = df[FEATURE_COLS]
y = df[TARGET_COLS]

print("Feature matrix shape:", X.shape)
print("Target matrix shape:", y.shape)


Feature matrix shape: (2832, 89)
Target matrix shape: (2832, 3)


Split Data for Training and Validation

In [4]:
import numpy as np

split_idx = int(len(X) * 0.8)
X_train, X_val = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_val = y.iloc[:split_idx], y.iloc[split_idx:]

print("Training shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_val.shape, y_val.shape)


Training shape: (2265, 89) (2265, 3)
Validation shape: (567, 89) (567, 3)


In [10]:
pip install --upgrade scikit-learn


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import os
import numpy as np

# ---- Ensure models directory exists ----
os.makedirs("models", exist_ok=True)

# ---- Initialize Random Forest Regressor ----
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

# ---- Train the model ----
rf_model.fit(X_train, y_train)

# ---- Predict on validation set ----
y_pred_rf = rf_model.predict(X_val)

# ---- Evaluate ----
# Use version-independent RMSE calculation
rmse_rf = np.sqrt(mean_squared_error(y_val, y_pred_rf))
mae_rf = mean_absolute_error(y_val, y_pred_rf)
r2_rf = r2_score(y_val, y_pred_rf)

print(f"Random Forest - RMSE: {rmse_rf:.3f}, MAE: {mae_rf:.3f}, R²: {r2_rf:.3f}")

# ---- Save the trained model ----
model_path = "models/rf_aqi_model_v1.pkl"
joblib.dump(rf_model, model_path)
print(f"[INFO] Random Forest model saved to {model_path}")


Random Forest - RMSE: 14.462, MAE: 11.788, R²: -0.247
[INFO] Random Forest model saved to models/rf_aqi_model_v1.pkl


In [ ]:
pip show scikit-learn


Name: scikit-learn
Version: 1.8.0
Summary: A set of python modules for machine learning and data mining
Home-page: 
Author: 
Author-email: 
License: 
Location: c:\Users\NAWAB AHMAD\aqi-forecasting-mlops\venv\Lib\site-packages
Requires: joblib, numpy, scipy, threadpoolctl
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [16]:
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

# ---- Initialize ----
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

# ---- Train ----
xgb_model.fit(X_train, y_train)

# ---- Predict ----
y_pred_xgb = xgb_model.predict(X_val)

# ---- Evaluate (version-safe) ----
rmse_xgb = np.sqrt(mean_squared_error(y_val, y_pred_xgb))
mae_xgb = mean_absolute_error(y_val, y_pred_xgb)
r2_xgb = r2_score(y_val, y_pred_xgb)

print(
    f"XGBoost - RMSE: {rmse_xgb:.3f}, "
    f"MAE: {mae_xgb:.3f}, "
    f"R²: {r2_xgb:.3f}"
)

# ---- Save Model ----
joblib.dump(xgb_model, "models/xgb_aqi_model_v1.pkl")


XGBoost - RMSE: 15.839, MAE: 12.995, R²: -0.498


['models/xgb_aqi_model_v1.pkl']

RIDGE REGRESSION

In [18]:
from sklearn.impute import SimpleImputer

# ---------------- Imputer ----------------
imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)


In [21]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

ridge_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])

# Train
ridge_pipeline.fit(X_train, y_train)

# Predict
y_pred_ridge = ridge_pipeline.predict(X_val)

# Evaluate
rmse_ridge = np.sqrt(mean_squared_error(y_val, y_pred_ridge))
mae_ridge = mean_absolute_error(y_val, y_pred_ridge)
r2_ridge = r2_score(y_val, y_pred_ridge)

print(f"Ridge - RMSE: {rmse_ridge:.3f}, MAE: {mae_ridge:.3f}, R²: {r2_ridge:.3f}")


Ridge - RMSE: 20.082, MAE: 16.672, R²: -1.429
